# Change of Variables

Chapter 13: Probability and Random Variables · *Mathematics for AI and Machine Learning* by Fuheng Wu

Uniform→Exponential (1D) and linear pushforward of a bivariate Gaussian with Jacobian factor.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-13/change-of-variables/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch
from torch.distributions import MultivariateNormal, Normal

torch.manual_seed(0)

# ── 1D: X ~ Uniform(0, 1), Y = -log X  ->  Y ~ Exponential(1) ──────────────
x0 = 0.3                                         # the demo's default x
y0 = -math.log(x0)
# f_Y(y) = f_X(g^-1(y)) |d g^-1 / dy| = 1 * e^{-y}
print(f"x = {x0}: y = {y0:.4f}, f_Y(y) = e^-y = {math.exp(-y0):.4f}")
y = -torch.rand(500000, dtype=torch.float64).log()
print("samples of -log U: mean", y.mean().item(), "(1), P(Y > 1) =", (y > 1).double().mean().item(),
      "(e^-1 =", math.exp(-1), ")")

# ── 2D: Y = A Z + b with Z ~ N(0, I)  ->  Y ~ N(b, A A^T) ──────────────────
A = torch.tensor([[2.0, 1.0],
                  [0.0, 1.0]], dtype=torch.float64)    # the demo's default a11, a12, a22
b = torch.tensor([1.0, 2.0], dtype=torch.float64)
base = MultivariateNormal(torch.zeros(2, dtype=torch.float64), torch.eye(2, dtype=torch.float64))


def log_density_Y(y):
    """log p_Y(y) = log p_Z(A^-1 (y - b)) - log |det A|."""
    z = torch.linalg.solve(A, y - b)
    return base.log_prob(z) - torch.linalg.det(A).abs().log()


target = MultivariateNormal(b, A @ A.T)
pt = torch.tensor([1.5, 2.5], dtype=torch.float64)
print("\n|det A| =", torch.linalg.det(A).abs().item(), "(areas grow by this factor, density shrinks by it)")
print("change-of-variables log p_Y =", log_density_Y(pt).item())
print("N(b, A A^T).log_prob        =", target.log_prob(pt).item())

samples = base.sample((300000,)) @ A.T + b
print("sample covariance =", torch.cov(samples.T).tolist(), " A A^T =", (A @ A.T).tolist())

x = 0.3: y = 1.2040, f_Y(y) = e^-y = 0.3000
samples of -log U: mean 1.002446665595152 (1), P(Y > 1) = 0.368468 (e^-1 = 0.36787944117144233 )

|det A| = 2.0 (areas grow by this factor, density shrinks by it)
change-of-variables log p_Y = -2.6560242469692907
N(b, A A^T).log_prob        = -2.6560242469692907
sample covariance = [[5.000081748487593, 0.9915452878281589], [0.9915452878281589, 0.995537072467977]]  A A^T = [[5.0, 1.0], [1.0, 1.0]]
